# PageIndex（vectorless RAG）

[VectifyAI/PageIndex](https://github.com/VectifyAI/PageIndex)（MIT License）

ベクトルDBもchunkingも使わず、文書を**目次のような木構造のインデックス**に変換し、LLMがその木を推論（reasoning）でたどって該当箇所を探す「vectorless, reasoning-based RAG」。人間の専門家が長いレポートの目次を見て該当章を開いて読む動きを模倣している（AlphaGoの木探索に着想を得たとのこと）。

## 仕組み

1. **インデックス構築**：文書のレイアウトから階層構造（章・節）を抽出してノードを作る。構造抽出自体はLLMを使わず、LLMは各ノードの要約・整形のみを担当する
    - ノードは `node_id`, `title`, `summary`, `start_index`/`end_index`（ページ範囲）, `sub_nodes`（子ノード）などを持つJSON
2. **検索（tree search）**：以下をループする
    1. LLMが木（タイトルと要約）を読む
    2. 有望なノードを選ぶ
    3. 選んだノードの本文を読む
    4. 回答に十分な情報が集まったか判定し、不足なら1へ戻る

会話履歴を踏まえて検索できる点や、「Appendix G参照」のような文書内の相互参照をたどれる点も利点として挙げられている。

## ベクトルRAGの問題点として主張していること

- クエリは「意図」を表すもので「内容」とは一致しない（query-knowledge mismatch）
- 意味的な類似度（similarity）が高くても関連性（relevance）が高いとは限らない。金融・法務などの専門文書で顕著
- 固定長のchunkingは文や節を分断して文脈を失わせる
- 各クエリを独立に扱うため会話の流れを使えない
- 文書内の相互参照を拾えない

## 性能

- FinanceBenchで正答率98.7%（ベクトルRAGは約50%）と主張。ただし自社製品（Mafin 2.5）での数値で、第三者による再現は見当たらない
- インデックス構築コストは公称約$0.001/ページだが、第三者の試算ではGPT-4o使用時に100ページで$0.5〜5程度とされ、幅がある

## 弱点

- **遅い**：1クエリごとにLLMを複数回逐次的に呼ぶため数秒かかる（ベクトル検索はミリ秒単位）
- **コスト**：インデックス構築時と検索時の両方でLLMの費用がかかる
- **スケール**：長い構造化文書1件には強いが、大量の短い非構造文書の横断検索には向かない。複数文書を扱うには文書を選ぶ層が別途必要
- レイテンシやスループットの公式ベンチマークがない
- OSS版はテキストPDFのみ対応。OCRや画像理解はクラウド版（API）のみ

## 使いどころ

有価証券報告書・契約書・技術マニュアルなど、**長く階層が明確な文書1件に対する精密なQA**に向く。大量文書の横断検索や低レイテンシが必要な用途は従来のベクトルRAGが有利なので、「ベクトル検索で文書を絞り込み、PageIndexで文書内を探す」ようなハイブリッドが現実的と思われる。

参考：

- [Next-Generation Vectorless, Reasoning-based RAG - PageIndex](https://pageindex.ai/blog/pageindex-intro)
- [Vectorless RAG: PageIndex - GeeksforGeeks](https://www.geeksforgeeks.org/artificial-intelligence/vectorless-rag-pageindex/)
- [PageIndex Vectorless RAG: Good, Bad, and Ugly](https://sjramblings.io/pageindex-deep-dive-vectorless-rag/)
- [Proxy-Pointer RAG: Achieving Vectorless Accuracy at Vector RAG Scale and Cost](https://towardsdatascience.com/proxy-pointer-rag-achieving-vectorless-accuracy-at-vector-rag-scale-and-cost/)